# BMD-45 Annotation Probe (no images, CPU-only)

One question: does `BMD-45-Train` contain usable `motorcycle` / `truck` /
`bicycle` counts that HeTra lacked? The answer green-lights or kills the
Option B (aggressive merge + BMD-45) training run.

## Data flow
1. Download **annotations + split lists only** (`*.json`, `*.txt` — MBs,
   no images) into ephemeral `/tmp`.
2. Parse the COCO JSON: true category list, per-class counts, box geometry,
   Option A kept/DROPPED verdict per category.
3. Nothing leaves Colab except pasted text — paste cell 4's output back.

## Decision rule
- Healthy counts (thousands) for the 3 missing classes → Option B training.
- Absent/near-zero → probe UVH-26/DETRAC next, or accept 3-class coverage.

## Runtime instructions
1. `Runtime` -> CPU. `Restart and run all`.
2. Nothing to edit, nothing to download except pasted output.


## 1. Setup

Pinned downloader; the import assert fails fast on a bad environment.

In [1]:
!pip install -q "huggingface_hub==1.30.0"

import huggingface_hub
print('huggingface_hub:', huggingface_hub.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 12.6 MB/s eta 0:00:00
huggingface_hub: 1.30.0


## 2. Download (annotations only)

Never assume the interior layout: list first, resolve robustly, fail fast
with the real listing. `max_workers=2` keeps us under the 429 limit.

In [2]:
from huggingface_hub import snapshot_download
from pathlib import Path

HF_REPO = "kalyan1729/trafficmanagementdataset"
HF_SUBSET = "BMD-45-Train"
# annotations + split lists only: MBs, no images
ALLOW_PATTERNS = ["BMD-45-Train/*.json", "BMD-45-Train/*.txt",
                  "BMD-45-Train/**/*.json", "BMD-45-Train/**/*.txt"]

RAW_ROOT = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                             allow_patterns=ALLOW_PATTERNS, max_workers=2)

cands = [d for d in Path(RAW_ROOT).rglob('*')
         if d.is_dir() and d.name.lower() == HF_SUBSET.lower()]
print('subset candidates:', [str(d) for d in cands])
assert cands, ('subset ' + HF_SUBSET + ' not found; top level: '
               + str([p.name for p in sorted(Path(RAW_ROOT).iterdir())][:15]))
SUBSET_ROOT = Path(cands[0])
print('SUBSET_ROOT:', SUBSET_ROOT)
print('interior:', sorted(p.relative_to(SUBSET_ROOT).as_posix()
      for p in SUBSET_ROOT.rglob('*') if p.is_file())[:30])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

subset candidates: ['/root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd/BMD-45-Train']
SUBSET_ROOT: /root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd/BMD-45-Train
interior: ['_annotations.coco.json']


## 3. Analyze

Self-contained: every helper this cell needs is defined here, so re-running
it alone is always safe. Raw category names are preserved verbatim — HeTra
taught us not to trust the dataset card.

In [3]:
assert 'SUBSET_ROOT' in dir(), 'run cell 2 (Download) first'
import json
import pandas as pd
from pathlib import Path

# Contract mapping (mirrors scripts/prepare_dataset.py, Option A).
CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]
ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "two_wheeler": "motorcycle", "two-wheeler": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle", "cycle": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
    "three-wheeler": "auto", "threewheeler": "auto",
}

def contract_class(name):
    n = (name or "").lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return m if m in CLASSES else None

jsons = sorted(SUBSET_ROOT.rglob('*.json'))
print('annotation files:', [p.name for p in jsons])
assert jsons, f'no JSON under {SUBSET_ROOT}'
# Heaviest file first (usually the single _annotations.coco.json).
jsons.sort(key=lambda p: p.stat().st_size, reverse=True)
ann_path = jsons[0]
size_mb = ann_path.stat().st_size / 1e6
print(f'parsing {ann_path.name} ({size_mb:.1f} MB)')
assert size_mb < 2000, 'annotation file implausibly large, aborting'
data = json.loads(ann_path.read_text(encoding='utf-8'))

cats = {c['id']: c['name'] for c in data.get('categories', [])}
imgs = {im['id']: im for im in data.get('images', [])}
print(f"{len(cats)} categories, {len(imgs)} images, "
      f"{len(data.get('annotations', []))} annotations")
assert cats and imgs, 'empty categories/images — wrong JSON file?'

rows = []
for a in data.get('annotations', []):
    raw = cats.get(a.get('category_id'), 'UNKNOWN')
    cc = contract_class(raw)
    x, y, w, h = a['bbox']
    im = imgs.get(a.get('image_id'), {})
    iw, ih = im.get('width') or 1, im.get('height') or 1
    rows.append({'image_id': a.get('image_id'),
               'file': str(im.get('file_name', '')),
               'raw_class': raw, 'contract_class': cc if cc else 'DROPPED',
               'kept': cc is not None,
               'area': round((w / iw) * (h / ih), 5),
               'aspect': round((w / iw) / max(h / ih, 1e-9), 3)})
df = pd.DataFrame(rows)
assert len(df) > 0, 'zero annotation rows parsed'
print(f'parsed {len(df)} rows')


annotation files: ['_annotations.coco.json']
parsing _annotations.coco.json (49.4 MB)
13 categories, 35792 images, 373132 annotations
parsed 373132 rows


## 4. Paste-back summary

Copy this cell's full output back — it is the entire green-light input.

In [4]:
assert 'df' in dir(), 'run cell 3 (Analyze) first'
import pandas as pd
pd.set_option('display.width', 160)
print('=== raw categories x verdict ===')
ct = pd.crosstab(df['raw_class'], df['contract_class'], margins=True)
print(ct)
kept_share = df['kept'].mean()
print(f'\nOption A keeps {kept_share:.1%} of {len(df)} boxes')
print('\n=== target-class check (motorcycle / truck / bicycle) ===')
for t in ('motorcycle', 'truck', 'bicycle'):
    n = int((df['contract_class'] == t).sum())
    flag = 'USABLE' if n >= 1000 else ('THIN' if n > 0 else 'ABSENT')
    print(f'{t:<10} {n:>8} boxes  [{flag}]')
print('\n=== kept-box geometry quantiles ===')
print(df[df['kept']][['area', 'aspect']].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(4))
print(f"\nimages referenced: {df['image_id'].nunique()}")


=== raw categories x verdict ===
contract_class   DROPPED   auto  bicycle    bus  motorcycle  truck     All
raw_class                                                                 
Bicycle                0      0     4713      0           0      0    4713
Bus                    0      0        0  13899           0      0   13899
Hatchback          35101      0        0      0           0      0   35101
LCV                18115      0        0      0           0      0   18115
MUV                 9092      0        0      0           0      0    9092
Mini-bus            1439      0        0      0           0      0    1439
SUV                15573      0        0      0           0      0   15573
Sedan              19970      0        0      0           0      0   19970
Tempo-traveller     4060      0        0      0           0      0    4060
Three-wheeler          0  65899        0      0           0      0   65899
Truck                  0      0        0      0           0   9968 

## 5. Handoff

1. Paste cell 4's output back.
2. Verdict lanes: 3 × USABLE → build Option B merge + training config;
   anything else → UVH-26/DETRAC probe or accept 3-class coverage.